____
### 1. Imports

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

import torch as torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal


____
### 2. Environment

Creating a custom environment to simulate the following condtions:
1. 30 day period
2. Required to sell inventory of 100 units
3. Unit cost of each unit set at $5
4. Agents in the environment are required to maximise profit in the 30 day period


Methods (1 - 3 are required for all environments):
1. `__init__()` → setup
2. `reset()` → start a new episode
3. `step(action)` → apply an action
4. `demand(price)` → calculates the demand given a price

Attributes:
1. observation space, 5 numbers as an array [Leftover Stock, Days Left, Last known demand]
2. action space (price of good, number over a continuouse range from 5 to 50)
3. max steps (days of simulation)
4. cost (cost incurred to obtain 1 unit)


In [ ]:
class DynamicPricingEnv(gym.Env):
    def __init__(self):
        self.max_steps = 30
        self.max_inventory = 100
        self.cost = 5.0
        self.observation_space = gym.spaces.Box( #observation is given by [inventory, days left, number of units sold last]
            low=np.array([0, 0, 0]),
            high=np.array([self.max_inventory, self.max_steps, self.max_inventory]),
            dtype=np.float32
        )
        self.action_space = gym.spaces.Box(low=5.0, high=50.0, shape=(1,), dtype=np.float32)
        #action is a discrete number between 5 to 50
        self.latest = np.array([self.max_inventory, self.max_steps, 0], dtype=np.float32)

    def reset(self, seed=None, options=None): #resets the values in the state
        self.inventory = self.max_inventory 
        self.step_count = 0
        self.last_demand = 0
        self.latest = self.obs()
        return self.obs(), {} #extra info dictionary is empty

    def step(self, action):
        price = float(action[0]) #obtain the price as a float
        demand = self.demand(price) #calculating demand from the price set
        units_sold = min(demand, self.inventory) #sell out 

        self.inventory -= units_sold #updated inventory
        self.step_count += 1 # step + 1
        self.last_demand = units_sold # updating last known demand

        reward = (price - self.cost) * units_sold #reward is the profit generated from that round
        terminated = self.inventory <= 0
        truncated = self.step_count >= self.max_steps
        self.latest = self.obs()
        return self.obs(), reward, terminated, truncated, {}

    def obs(self): #to obtain observation of the current state
        return np.array([self.inventory, self.max_steps - self.step_count,
                         self.last_demand], dtype=np.float32)

    def get_latest(self):
        obs = self.latest
        print(f"Leftover Stock: {obs[0]} units, Days Left {obs[1]}, Sold Units: {obs[2]}")

    def demand(self, price):
        base = 40
        sensitivity = 1.2
        noise = np.random.normal(0, 3) #noise is random number from 0 to 3
        return int(max(0, base - sensitivity * price + noise))

____
#### 2.1 Exploring the environment

In [ ]:
env = DynamicPricingEnv()

In [ ]:
obs, info = env.reset()
print(f"Observation space representing: [stock left, days left, last known sold]: {obs}")
print(f"Extra info dictrionary: {info}")

obs, reward, terminated, truncated, info = env.step([10.0])
env.get_latest()
print(reward)
print(f"{terminated}, {truncated}")

____
### 3. Deciding which model to use 
- A standard Q-table cannot be used as the action space (price of good) is a continuous number instead of a discrete number

#### 3.1 Proximal Policy Optimization (PPO) Model
- a policy gradient method which directly learns "Given this state, what price should I output"
- the "Proximal" part of the PPO model adjusts the actions in small amounts to find the optimal policy 
- therefore, PPO Models limits how drastically the policy changes each update to prevent unstable training

##### PPO Architecture
- PPO uses 2 networks
1. Actor Network
    - Outputs the pricing policy
    - Given a state, feeds into a neutal network and outputs a pricing distribution
    - `state → neural network → price distribution`
    - The agent then samples prices around that range

2. Critic Network
    - Estimates the future rewards
    - Asks "How profitable is this situation" and helps the Actor Network improve

##### PPO Flow
`Observe market → Choose price → Simulate customer response → Get profit reward → Update pricing policy slightly`


#### 3.2 Twin Delayed Deep Deterministic Policy Gradient (TD3)
- Designed specefically for continuous action space, precise control and stable deep Q-learning
- instead learning "what action should I take?" the model learns "how good is a particular action"

##### TD3 Architecture
- TD3 uses 3 networks
1. Actor Network
    - Outputs a distribution over actions
    - `state → distribution`
    - Samples from the distribution to create exploration

2. Critic Network (2 critic networks)
    - Each critic network estimates a Q-value, which represents the total future reward if an action is taken in this state, ie `Q(state, action)` (same as TD3)

3. Replay Buffer
    - To store experiences and reuse them, making the SAC highly sample efficient
    - Allow SAC to learn from past experiences repeatedly

4. Target Networks

##### TD3 Flow
- `Observe State → Actor suggests an action → Twin Critic Networks evaluate long term reward of the actions → Lower Q-value generated from the 2 networks is used`
- Therefore, the lower Q-value is used to train the critics and the actor (actor updated occassionally)



#### 3.3 Soft Actor-Critic (SAC)
- Considered one of the strongest RL algorithms for continuous control
- Combines actor-critic learning, entropy maximisation and off-policy training
- Tries to maximise both `Reward` and `Exploration` instead of only `profit`
- Entropy refers to the epsilon (randomness of the actions) therefore it encourages the agent to keep exploring pricing options, preventing the model from becoming too deterministic 


##### SAC Architecture
- SAC uses 3 networks
1. Actor Network
    - Outputs the exact price
    - `state → price`

2. Critic Network (2 critic networks)
    - The "Twin" part is in refernce to the 2 critic networks
    - Each critic network estimates a Q-value, which represents the total future reward if an action is taken in this state, ie `Q(state, action)` 

##### SAC Flow
- `Observe State → Sample action from policy distribution → receive reward → update critic → update actor → encourage exploration through entropy bonus`
- Reward is evaluated as `total reward = reward + entropy bonus` and to encourage exploration

____
### 4. Training the PPO Model
- In the spirit of learning, I will be training a PPO model first before training a TD3 model and a SAC model

#### 4.1 Establishing Architecture
- Actor and Critic networks are first created
    - Actor and Critic networks are used to observe the state and output the action and critic values, no learning logic implemented yet
- Create RolloutBuffer to store data and translate the data into learning signals 
    - Data is converted to learning signals, networks do not learn yet
- `ppo_update()` function converts teh learning signals and updates the networks via gradient descent
    - computes policy loss, value loss and entropy loss
- `train()` function combines the network, rollout buffer and `ppo_update()` to train the model 

#### 4.2 Creating Actor and Critic Networks

In [ ]:
class ActorCritic(nn.Module): # neural network defined as a PyTorch module, inheriting from nn.module
    def __init__(self, obs_dim, action_dim):
        super().__init__() #initialises the parent nn.Module class
        # self.backbone acts as a shared extractor used by both the actor and critic network
        self.backbone = nn.Sequential( #nn.Sequential runs the layers in order, linear -> Tanh -> linear -> tanh
            nn.Linear(obs_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
        )
        self.actor_mean = nn.Linear(64, action_dim) 
        self.log_std = nn.Parameter(torch.zeros(action_dim)) 
        self.critic = nn.Linear(64, 1)

    def forward(self, obs): #needs to be overridden
        features = self.backbone(obs) #vector of 64 values
        mean = self.actor_mean(features) #obtain the action 
        std = self.log_std.exp().expand_as(mean) #obtain the std dev and fits the shape with with mean 
        critic_val = self.critic(features) #obtain the critic value
        return mean, std, critic_val
    
    def get_action(self, obs): #creating action based off the network
        mean, std, value = self.forward(obs) #calling forward to obtain values from actor and critic network
        dist = Normal(mean, std) #creates normal distribution
        action = dist.sample() #samples the distribution
        log_prob = dist.log_prob(action).sum(dim=-1) #obtains sum of log distribution (exp below)
        return action, log_prob, value.squeeze(-1) #converts the value into a scalar

    def evaluate(self, obs, action):
        mean, std, value = self.forward(obs)
        dist = Normal(mean, std)
        log_prob = dist.log_prob(action).sum(dim=-1)
        entropy = dist.entropy().sum(dim=-1) #calculates the entropy of the normal distribution, how random or uncertain the distribution is 
        # entropy is summed up over the last dimension
        return log_prob, value.squeeze(-1), entropy

##### 4.21 Explanation of Code:
```python
self.backbone = nn.Sequential( 
            nn.Linear(obs_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh()
        )
```
- `nn.sequential()` ensures that the following layers run in sequence
- `nn.Linear(obs_dim, 64)` -> y = Wx + b
    - x is a `3x1 vector` since the observation is a 3-dimensional vector with 3 values
    - W is a `64x3 matrix` such that the product `Wx` is a 64-dimensional vector with 64 values
    - b is a `64x1 vector` bias function that weighs down each value inside the produce `Wx`
    - y is the resultant `64x1 vector`, representing the first layer of the neural network, where each number of the vector corresponds to a neuron
- `nn.Tanh()` -> y = tanh(x)
    - x is a `64 x 1 vector` that is the result from the linear vector
    - y is the resultant `64 x 1 vector` from applying the tanh() function on every single value in the original vector
    - this function introduces non-linearlity and squashes every value to be within the range (-1, 1)
    - the introduction of non-linearity allows the model to learn non-linear behaviours
- `nn.Linear(64, 64)` -> y = Wx + b
    - uses the non-linear outputs from the previous layers but turns them into more sophisticated outputs using weights and biases
- `nn.Tanh()` -> y = tanh(x)
    - squishes all values to within the range (-1, 1) and introduces non-linearity to the outputs of the previous layer

```python
self.actor_mean = nn.Linear(64, action_dim) 
self.log_std = nn.Parameter(torch.zeros(action_dim))
self.critic = nn.Linear(64, 1)
```
- `self.actor_mean = nn.Linear(64, action_dim)`
    - Makes use of the 64 outputs from the backbone to derive the outputs in the action dimension (mean of the price)
- `self.log_std = nn.Parameter(torch.zeroes(action_dim))`
    - std deviation measures the randomness of the distributionm, log(std) is used so that the value is not -ve, later converted using `exp()`
    - wrapping it in `nn.Parameter()` means that the value should be learning during training
    - this makes it such that the PPO model learns during training what is the appropriate level of exploration
- `self.critic = nn.Linear()`
    - Makes use of the 64 outputs from the backbone to derive 1 value, which represents the future reward expected from the current state

- `log_prob = dist.log_prob(action).sum(dim=-1)`
    - `dist.log_prob(action)` obtains the log_probability of the action within the distribution
    - `.sum(dim=-1)` sums it along the last dimension
    - instad of multiplying individual probabilities, it adds up `log(prob)` instead 

##### 4.22 Over-Arching View:
- observation (vector of 3 values) -> self.backbone(vector of 64 values) 
- The observation in the form of 64 values is then fed into the actor network and critic network
- ie `self.backbone(vector of 64 values) -> action (1 value)` and `self.backbone(vector of 64 values) -> future reward expected (1 value)`

##### 4.3 Creating RolloutBuffer
- Acts as a container to store past experiences and convert them into learning signals
- `RolloutBuffer` converts raw experiences into 3 learning signals:
    1. Advantages: How much better an action was than expected
    2. Returns: The total expected future reward
    3. Normalized Advantages: Standardized advantages for stable training

In [ ]:
class RolloutBuffer:
    def __init__(self):
        self.clear() #delegates initialisation to the clear() method

    def clear(self): #resets all lists to empty
        self.obs, self.actions, self.log_probs = [], [], []
        self.rewards, self.values, self.dones  = [], [], []

    def add(self, obs, action, log_prob, reward, value, done): #appends one round of experiences
        self.obs.append(obs)
        self.actions.append(action)
        self.log_probs.append(log_prob)
        self.rewards.append(reward)
        self.values.append(value)
        self.dones.append(done)

    def compute_returns(self, last_value, gamma=0.99, gae_lambda=0.95): #compute generalized advantage est
        advantages = [] 
        gae = 0.0
        values = self.values + [last_value] # adds one extra value to compute next-step difference
        for t in reversed(range(len(self.rewards))):
            delta = self.rewards[t] + gamma * values[t + 1] * (1 - self.dones[t]) - values[t]
            gae   = delta + gamma * gae_lambda * (1 - self.dones[t]) * gae
            advantages.insert(0, gae)
        returns = [adv + val for adv, val in zip(advantages, self.values)]
        return advantages, returns

    def to_tensors(self, advantages, returns, device):
        obs = torch.tensor(np.array(self.obs), dtype=torch.float32).to(device)
        actions = torch.stack(self.actions).to(device)
        log_probs = torch.stack(self.log_probs).to(device)
        advantages = torch.tensor(advantages, dtype=torch.float32).to(device)
        returns = torch.tensor(returns, dtype=torch.float32).to(device)
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        return obs, actions, log_probs, advantages, returns

##### 4.31 Explanation of code

```python
def compute_returns(self, last_value, gamma=0.99, gae_lambda=0.95):
    advantages = [] 
    gae = 0.0
    values = self.values + [last_value] 
    for t in reversed(range(len(self.rewards))): 
        delta = self.rewards[t] + gamma * values[t + 1] * (1 - self.dones[t]) - values[t]
        gae = delta + gamma * gae_lambda * (1 - self.dones[t]) * gae
        advantages.insert(0, gae)
    returns = [adv + val for adv, val in zip(advantages, self.values)]
    return advantages, returns
```
- `compute_returns()` is computes the Advantages ("How much better an action was than expected") and the returns ("total expected future rewards")
- Temporal Diff (TD) error: `delta = self.rewards[t] + gamma * values[t + 1] * (1 - self.dones[t]) - values[t]` measures the diff in outcome compared to the expected outcome (measuring suprise)
- Equation generalises to `TD = (actual outcome) - (expected outcome)`
    - delta < 0, worse than expected, delta > 0, better than expected
    - `self.rewards[t]` is the immediate reward from action 
    - `gamma` is the discount factor, measures how much the model cares about future rewards
    - `values[t + 1]` is the predicted future value, the critic's estimate of future reward after 1 step 
    - `gamma * values[t + 1]` is the expected future rewards, multiplying by `gamma` discounts the expected future rewards 
    (expected future rewards are slightly less vaulable)
    - `values[t]` is baseline prediction before seeing the outcome
    - `1 - self.dones[t]` acts as a switch, if the episode has not terminated at the step t, `self.dones[t] = 0`, if it has terminated, `self.dones[t] = 1` and `1 - self.dones[t] = 0`
    - When episode has ended, `1 - self.dones[t] = 0` and future term of `gamma * values[t + 1] * (1 - self.dones[t]) = 0`
    - Therefore, `delta = self.rewards[t] - values[t]` since there is no more `expected future rewards` and the difference in expectation is simply the rewards up to that point, `self.rewards[t]`, minus expected rewards `values[t]`

- Generalized Advantage Estimation: `gae = delta + gamma * gae_lambda * (1 - self.dones[t]) * gae` which is a smoothened estimate of "how good was this action"
- Equation generalises to `gae = current TD error + discounted future gae`
    - starts out with the current step's TD error
    - adds the discounted future TD errors = `gamma * gae_lambda * (1 - self.dones[t]) * gae`
    - when the episode ends, `self.dones[t] = 1` and `1 - self.dones[t] = 0` and the future TD errors = 0 (no more future for the episode since it terminated) 

- `advantages.insert(0, gae)`
    - inserts the calculated gae at the **front** of the `advantages` array
    - when the loop runs in reverse during `for t in reversed(range(len(self.rewards))):` the first iteration takes the last timestep. with the calculated gae inserted at the front, the advantages are in forward order

- `returns = [adv + val for adv, val in zip(advantages, self.values)]` : equivalent to doing:
    ```python
    returns = []
    for adv, val in zip(advantages, self.values):
        r = adv + val
        returns.append(r)
    ```
    - for each step, returns = advantage + value estimate
    - adding it to the array saved as returns 

``` python
def to_tensors(self, advantages, returns, device):
        obs = torch.tensor(np.array(self.obs), dtype=torch.float32).to(device)
        actions = torch.stack(self.actions).to(device)
        log_probs = torch.stack(self.log_probs).to(device)
        advantages = torch.tensor(advantages, dtype=torch.float32).to(device)
        returns = torch.tensor(returns, dtype=torch.float32).to(device)
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        return obs, actions, log_probs, advantages, returns
```
- `torch.tensor(data).to(device)`
    - Creates a new tensor from python data (list, arrays, numbers), `data -> tensor`
    - used for `obs`, `advantages` and `returns` as they are all raw lists of data
-  `torch.stack(tensors, dim = 0)`
    - joins multiple existing tensors along a new dimension
    - used for `actions` and `log_probs` as `self.actions` and `self.log_probs` is a list of tensors already
- `to(device)` is used to move a tensor to a specified device (CPU or GPU)
- `advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)`
    - `advantages - advantages.mean()` subtracts the mean from every value such that it is values in the tensors are now **centred around zero** 
    - `/(advantages.std() + 1e-8)` divides every value from the std deviation 
    - this process normalises all data within the tensor (tensor supports element-wise operations)


##### 4.32 Over-Arching View:
1. `add()` method
   - raw data is collected: observations, actions, log probabilities, rewards, values, and done flags
   - All stored as lists in the buffer

2. `compute_returns()` method
   - Compute TD errors (`delta`) by comparing actual outcomes to critic predictions
   - Accumulate GAE backward through time to get smooth advantage estimates
   - Combine advantages with value estimates to get target returns for the critic

3. `to_tensors()` method
   - Convert all lists to PyTorch tensors
   - Normalize advantages to have mean=0 and std=1 for stable training
   - Move tensors to the correct device (CPU or GPU)

4. **Learning Signal Output**
   - `advantages`: How much better/worse each action was compared to expected (used by actor)
   - `returns`: Target value estimates (used by critic)
   - Both aligned with original observations and actions for supervised learning

##### 4.4 Instantiate functions to update PPO

In [ ]:
def ppo_update(model, optimizer, obs, actions, old_log_probs,advantages, returns, 
               clip_range=0.2, ent_coef=0.01, vf_coef=0.5, n_epochs=10, batch_size=64):
    total_steps = obs.shape[0] 
    for _ in range(n_epochs):
        indices = torch.randperm(total_steps)
        for start in range(0, total_steps, batch_size):
            idx = indices[start : start + batch_size]
            new_log_probs, values, entropy = model.evaluate(obs[idx], actions[idx])
            ratio = (new_log_probs - old_log_probs[idx]).exp()
            adv = advantages[idx]
            policy_loss = -torch.min(
                ratio * adv,
                torch.clamp(ratio, 1 - clip_range, 1 + clip_range) * adv
            ).mean()
            value_loss = nn.functional.mse_loss(values, returns[idx])
            entropy_loss = -entropy.mean()
            loss = policy_loss + vf_coef * value_loss + ent_coef * entropy_loss
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
            optimizer.step()

##### 4.41 Explanation of code
- Required Parameters:
    1. `model` : ActorCritic neural network being trained
    2. `optimizer` : The optimizer that applies weight updates via backpropagation (adjusting weights of each matrix using backprop)
    3. `obs` : observations collected during rollout, `512x3 matrix`, converted to tensor (3 cols as observation space is 3 dimension)
    4. `actions` : actions taken during rollout, `512x1 vector`, converted to tensor (1 col as action space is just 1 value)
    5. `old_log_probs` : log prob of actions under the old policy (before update), used to compute importance sampling ratio
    6. `advantages` : normalized advantage estimates (array with 512 entries), tells us how good each action was compared to expected 
    7. `returns` : target returns for the critic (array with 512 entries), what the critic model should predict
- 512 -> max of 512 timesteps in an episode
- Hyperparameters:
    1. `clip_range=0.2` : clipping parameter, restricting the policy ration to [1-0.2, 1+0.2] = [0.8, 1.2], to prevent drastic policy changes
    2. `ent_coef=0.01` : weight on exploration bonus, provides incentive to explore 
    3. `vf_coef=0.5` : value function coefficient, the weight on critic loss - How important is predicting returns accurately
    4. `n_epochs=10` : number of passes through the data, extracting more learning signals each epock
    5. `batch_size=64` : batch size for gradient updates, process 64 timesteps at a time
    
- `total_steps = obs.shape[0]` 
    - `.shape` gives (rows, cols), therefore `obs.shape[0]` = rows of data = number of timesteps

```python
for _ in range(n_epochs):
    indices = torch.randperm(total_steps)
    for start in range(0, total_steps, batch_size):
```
- iterate through set number of times according to hyperparameters set
`indices = torch.randperm(total_steps)`
    - creates a tensor of the numbers from 0 to total_steps-1 in a **random order**
    - this shuffles the rollout data before training for each iteration so that the PPO does not see the data in the same order each time
    - this breaks any ordering bias and makes mini-batches random
`for start in range(0, total_steps, batch_size)`
    - iterates from 0 to total_steps, jumping by `batch_size` at once


- `idx = indices[start : start + batch_size]` : slices the array in chunks of length equal to `batch_size`, becomes the current batch fof sample indices
- `new_log_probs, values, entropy = model.evaluate(obs[idx], actions[idx])` : evaluates the model based off the small batch selected
- `ratio = (new_log_probs - old_log_probs[idx]).exp()` : computes the PPO probability ratio, measuring how much the new policy changed compared to the old policy for those same actions (element wise operation for the whole array)
    - ratio = 1, policy is unchanged
    - ratio > 1, the new policy assigns higher probability to the action ()
    - ratio < 1, the new policy assigns lower probability to the action
- `adv = advantages[idx]` : obtains the advantage values for this batch of values
- ``` python 
    policy_loss = -torch.min(
                ratio * adv,
                torch.clamp(ratio, 1 - clip_range, 1 + clip_range) * adv
            ).mean()
    ```
- `ratio * adv` : element-wise operation between importance-sampling ratio and the advantage array for this batch, `adv`
- `torch.clamp(ratio, 1 - clip_range, 1 + clip_range)` : `torch.clamp(tensor, min, max)` clips all the values inside the ratio tensor to within the range of (1 - clip_range, 1 + clip_range)
    - if value < min or value > max , the value is set to min / max
    - `* adv` to perform element-wise operation between importance-sampling ratio and the advantage array for this batch
- `torch.min(tensor1, tensor2)` : compares the 2 tensors element-wise and extracts the lower value at each position, constructing a new tensor out of it
- `.mean()` : averages all values of the tensor to obtain the mean 
- calculating policy loss:
    - finding the **mean advantage value** after multiplying by ratio element-wise and then **limiting it** to within the range `(1-clip_range, 1+clip_range)`
    - multiplying by `-` then obtains the `policy loss` 

- `value_loss = nn.functional.mse_loss(values, returns[idx])` : the Mean Square Error between critic predictions and targets
    - `nn.functional.mse_loss(input_tensor, target_tensor)` finds the mean squared error between every value in the input and the target tensor
    - use `reduce = none` to get the output as a tensor, else it returns the average of all the MSE values
- `entropy_loss = -entropy.mean()` : multiplies the mean entropy by -1, turning the maximise entropy into a minimization objective
    - when entropy increases, loss decreases which is what gradient descent wants
- `loss = policy_loss + vf_coef * value_loss + ent_coef * entropy_loss`
    - formula generalises to : overall loss = policy loss + overall value loss + overall entropy loss 
    - `overall value loss = vf_coef * value_loss`, where `vf_coef` scales the critic term in thte total loss equation (how strongly gradient descent prioritizes reducing the value, critic's accuracy, relative to policy and entropy)
    - `overall entropy loss = ent_coef * entropy_loss`, where `ent_coef` controls how strongly exploration is encouraged (how strongly gradient descent prioritizes exploration, relative to policy and value)
- `optimizer.zero_grad()` : clears the gradients stored inside the optimizer before calculations are done 
- `loss.backward()` : with loss represented as a function of policy loss, value loss and entropy loss, ie L = f(p,v,e)
    - calling `.backward()` differentiates loss wrt. p, v and e and stores it inside each parameter, which can be accessed with `.grad()`
- `nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)` : rescales `.grad()` in place if the total norm > max_norm, so that the parameter update magnitude is limited when `optimizer.step()` runs
- `optimizer.step()` : computes updates for each parameter's rule within the optimizer and updates it
    - this thus makes the new weight value of the neural network the model's current parameters    
    - this updates the whole network, meaning the backbone network, actor and critic network.

##### 4.42 Over-Arching View:
- The env produces 3‑D observations that a shared backbone encodes into features consumed by an `ActorCritic` network (actor outputs a Gaussian price policy; critic predicts state value).
- Episodes are collected into a `RolloutBuffer` (obs, actions, log-probs, rewards, values, dones); GAE computes advantages and returns from those rollouts.
- `ppo_update()` runs multiple epochs of minibatch updates with the clipped policy objective, value MSE, and entropy bonus to update the network.
- Repeat collect → compute (advantages/returns) → update until total timesteps, producing a trained pricing policy.

##### 4.5 Instantiate function to train model

In [ ]:
def train(total_timesteps=200_000, n_steps=512):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on: {device}")
    env = DynamicPricingEnv()
    obs_dim = env.observation_space.shape[0] # 3
    action_dim = env.action_space.shape[0] # 1
    #Adam (Adaptive Moment Estimation) is the optimizer that updates the network weights, maintaining 2 running statistic per parameter registered
    model = ActorCritic(obs_dim, action_dim).to(device) # moves all of the model's tensors to the device
    optimizer = optim.Adam(model.parameters(), #register all params, backkbone, actor head, log_std, critic
                            lr=3e-4) # learning rate set at 3e-4
    buffer = RolloutBuffer()
    obs, _ = env.reset()
    episode_reward = 0
    episode_count = 0
    timestep = 0
    while timestep < total_timesteps: # runs until timestep budget is exhausted
        buffer.clear() #reset the buffer at the start of every rollout: old experience thrown away as they were collected under a previous version of the policy
        for _ in range(n_steps):
            obs_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(device)
            with torch.no_grad(): #actions within this block temporarilty disables autograd and do not track history or build computation graphs
                action, log_prob, value = model.get_action(obs_tensor)
            action_np = action.cpu().numpy()[0]
            action_np = np.clip(action_np, 5.0, 50.0) 
            next_obs, reward, terminated, truncated, _ = env.step(action_np)
            done = terminated or truncated
            buffer.add(obs = obs,
                       action = action.squeeze(0).cpu(),
                       log_prob = log_prob.squeeze(0).cpu(),
                       reward = reward,
                       value = value.squeeze(0).cpu().item(),
                       done = float(done))
            episode_reward += reward
            obs = next_obs
            timestep += 1
            if done:
                episode_count += 1
                if episode_count % 20 == 0:
                    print(f"Timestep {timestep:>7} | Episode {episode_count:>4} | "
                          f"Reward: {episode_reward:>8.2f}")
                episode_reward = 0
                obs, _  = env.reset()
        with torch.no_grad():
            last_obs = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(device)
            _, _, last_value = model.get_action(last_obs)
            last_value = last_value.squeeze(0).cpu().item()
        advantages, returns = buffer.compute_returns(last_value)
        obs_t, act_t, lp_t, adv_t, ret_t = buffer.to_tensors(advantages, returns, device)
        ppo_update(model, optimizer, obs_t, act_t, lp_t, adv_t, ret_t)
    # torch.save(model.state_dict(), "ppo_pricing.pth")
    print("Training complete.")
    return model


##### 4.51 Explanation of code

```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")
env = DynamicPricingEnv()
obs_dim = env.observation_space.shape[0] # 3
action_dim = env.action_space.shape[0] # 1
model = ActorCritic(obs_dim, action_dim).to(device) 
optimizer = optim.Adam(model.parameters(), lr=3e-4) 
```
- `model = ActorCritic(obs_dim, action_dim).to(device)`
    - instantiate a model using the `obs_dim` and `action_dim` of the environment
    - using `.to(device)` moves the tensor of the model to `device`

- `optimizer = optim.Adam(model.parameters(), lr = 3e-4)`
    - creates a **Adaptive Moment Estimation** optimizer, updating the network's weights
    - runs 2 running statistics per parameter:
        1. The mean of gradients
        2. The mean of squared gradients 
    - `model.parameters()` as an input : registers all parameters of the model in the optimizer
    - `lr = 3e-4` : learning rate for the PPO, cannot be too low or too high

``` python
while timestep < total_timesteps:
    buffer.clear()
```
- outer loop runs until the total timestep budget is exhausted, `buffer.clear()` resets the buffer at the start of every rollout
- This ensures that old experiences are thrown away as they are collected under a previous version of the policy and no longer valid for the current update

```python
for _ in range(n_steps):
    obs_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(device)
```
- `torch.tensor(obs, dtype=torch.float32)` : converts the NumPy array (obs) into a PyTorch Tensor
    - datatype set to `float.32` as that is the type used by neural networks
- `unsqueeze(0)` : inserts a dimension at postion 0
    - `nn.Linear` layers expect inputs shaped as `[batch_size, features]` while the `obs` tensor is shaped `[3]`
    - therefore, `.unsqueeze(0)` inserts a dimension at position 0, making it `[1, 3]`, a batch of size 1
    - this is required for matrix multiplication


```python
with torch.no_grad():
    action, log_prob, value = model.get_action(obs_tensor)
```
- `with torch.no_grad()`
    - disables gradient computation for the methods inside, which is `action, log_prob, value = model.get_action(obs_tensor)`
    - in context here, it disables computation of gradient for `model.get_action(obs_tensor)` because this portion is just meant for data collection
    - therefore, there is no need to update the model as it is simply used to make decisions 
    - this is more efficient for computation

```python
action_np = action.cpu().numpy()[0]
action_np = np.clip(action_np, 5.0, 50.0) 
next_obs, reward, terminated, truncated, _ = env.step(action_np)
done = terminated or truncated
```
- `action.cpu().numpy()[0]` : `.cpu().numpy()` moves the tensor to the cpu so that numy can access it and converts it to a numpy array
    - `[0]` is used to remove the batch dimension, converting it from an array shaped `[1, 1]` (batch of 1, dimension 1) to an array of shape `[1]`, a 1-element array containing the price
- `np.clip(5.0, 50.0)` : limits the action to be a price within 5 and 50 since the range is theoratically unlimited
    - clip the range after sampling rather than sampling from a bounded distribution
- `next_obs, reward, terminated, truncated, _ = env.step(action_np)` : obtains the state after the nn provides an action

```python
buffer.add(obs = obs,
            action = action.squeeze(0).cpu(),
            log_prob = log_prob.squeeze(0).cpu(),
            reward = reward,
            value = value.squeeze(0).cpu().item(),
            done = float(done))
```
- updating the buffer with all new states, applying `.squeeze(0)` to `action`, `log_prob` and `value` to adjust its shape and converts `done` to a float for computation purposes within buffer

```python
with torch.no_grad():
    last_obs = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(device)
    _, _, last_value = model.get_action(last_obs)
    last_value = last_value.squeeze(0).cpu().item()
advantages, returns = buffer.compute_returns(last_value)
obs_t, act_t, lp_t, adv_t, ret_t = buffer.to_tensors(advantages, returns, device)
```
- This step is to process any partial episodes at the end of the 512 timestep limit.
- e.g. if each episode take 100 steps, the last 12 steps will be an incomplete episode. Therefore, this step will process the last known observation 
    - This processing is done using `with torch.no_grad()` to turn off the computation of gradients so that the model is not updated using incomplete episode data
    - next two lines are then used to obtain the price suggested by the model 
- `advantages, returns = buffer.compute_returns(last_value)` : use the last observation, the incomplete data to calculate the advantages and returns
- `obs_t, act_t, lp_t, adv_t, ret_t = buffer.to_tensors(advantages, returns, device)` : converts the data stored in the buffer to learning signals 





In [ ]:
model = train(total_timesteps=200_000)

##### 4.15 Instantiate functions to evaluate model

In [ ]:
def evaluate_model(model, n_episodes=50, deterministic=True):
    device = next(model.parameters()).device
    model.eval()

    episode_rewards = []
    ending_inventory = []
    steps_taken = []

    for _ in range(n_episodes):
        env = DynamicPricingEnv()
        obs, _ = env.reset()
        done = False
        total_reward = 0.0
        while not done:
            obs_tensor = torch.tensor(obs, dtype=torch.float32, #convert state to a batched tensor 
                                      device=device).unsqueeze(0) #unsqueeze.(0) inserts a new dimension of size 1 at the 0th position
            with torch.no_grad(): #temporarily disables autograd so that operations inside do not track history or build computation graphs
                mean, std, _ = model.forward(obs_tensor)
                if deterministic:
                    action = mean
                else:
                    action = Normal(mean, std).sample()

            action_np = action.squeeze(0).detach().cpu().numpy()
            action_np = np.clip(action_np, 5.0, 50.0).astype(np.float32)

            obs, reward, terminated, truncated, _ = env.step(action_np)
            total_reward += reward
            done = terminated or truncated

        episode_rewards.append(total_reward)
        ending_inventory.append(env.inventory)
        steps_taken.append(env.step_count)

    results = {
        "episode_rewards": episode_rewards,
        "mean_reward": float(np.mean(episode_rewards)),
        "std_reward": float(np.std(episode_rewards)),
        "min_reward": float(np.min(episode_rewards)),
        "max_reward": float(np.max(episode_rewards)),
        "mean_ending_inventory": float(np.mean(ending_inventory)),
        "mean_steps": float(np.mean(steps_taken)),
    }

    print(f"Episodes: {n_episodes}")
    print(f"Mean reward: {results['mean_reward']:.2f} +/- {results['std_reward']:.2f}")
    print(f"Reward range: [{results['min_reward']:.2f}, {results['max_reward']:.2f}]")
    print(f"Mean ending inventory: {results['mean_ending_inventory']:.2f}")
    print(f"Mean steps per episode: {results['mean_steps']:.2f}")

    return results

In [ ]:
eval_results = evaluate_model(model, n_episodes=100, deterministic=True)